<a href="https://colab.research.google.com/github/sy3da/DA-V2-for-RobotNav/blob/yslin%2Fevaluate-metrics/evaluate_metrics_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DAV2 Navigation — Metrics Evaluation
**Computes Free-space IoU and Obstacle Precision/Recall on NYU-D (and optionally KITTI)**

Steps:
1. Verify GPU
2. Clone repo and install dependencies
3. Download DAV2 metric-depth checkpoint (ViT-S, Hypersim)
4. Download and extract NYU-D dataset
5. Run DAV2 metric-depth inference → save prediction `.npy` files
6. Evaluate metrics (Free-space IoU + Obstacle Precision/Recall)
7. (Optional) KITTI evaluation
8. Print final summary table and download results

---
**Important:** Run cells top-to-bottom in a single session. Do **not** restart the runtime between steps, as the model object is reused.

## 0. Verify GPU

In [ ]:
import torch
print(f'GPU available : {torch.cuda.is_available()}')
print(f'GPU name      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
# Expected: NVIDIA A100 (Colab Pro)

GPU available : True
GPU name      : Tesla T4


## 1. Clone Repo and Install Dependencies

In [ ]:
import os, sys

REPO_DIR = '/content/DA-V2-for-RobotNav'
BRANCH   = 'yslin/evaluate-metrics'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/sy3da/DA-V2-for-RobotNav.git {REPO_DIR}')

os.chdir(REPO_DIR)
os.system(f'git checkout {BRANCH}')
os.system(f'git pull origin {BRANCH}')
print('Repo ready at:', os.getcwd())

# IMPORTANT: insert metric_depth folder FIRST so the metric-depth version
# of DepthAnythingV2 (which accepts max_depth) takes priority over the
# root-level relative-depth version.
sys.path.insert(0, f'{REPO_DIR}/metric_depth')
sys.path.insert(1, REPO_DIR)
print('sys.path set correctly')

Repo ready at: /content/DA-V2-for-RobotNav
sys.path set correctly


In [ ]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'torch', 'torchvision',
    'opencv-python-headless',
    'h5py', 'numpy',
    'huggingface_hub',
    'Pillow'
])
print('Dependencies installed.')

Dependencies installed.


## 2. Download DAV2 Metric-Depth Checkpoint (ViT-S, Hypersim)

We use the **metric-depth** variant trained on Hypersim (indoor scenes).
This model outputs depth in **metres** (larger value = farther), which matches NYU-D ground truth units.

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)
CKPT = 'checkpoints/depth_anything_v2_metric_hypersim_vits.pth'

if not os.path.exists(CKPT):
    print('Downloading metric-depth ViT-S checkpoint (~100 MB)...')
    url = ('https://huggingface.co/depth-anything/'
           'Depth-Anything-V2-Metric-Hypersim-Small/resolve/main/'
           'depth_anything_v2_metric_hypersim_vits.pth')
    os.system(f'wget -q --show-progress "{url}" -O {CKPT}')
    print('Download complete.')
else:
    print('Checkpoint already exists, skipping.')

Download complete.


## 3. Download and Extract NYU-D Dataset

The dataset is hosted on HuggingFace as `.tar` archives containing `.h5` files.
We extract 100 RGB images and their ground-truth depth maps (in metres).

In [ ]:
from huggingface_hub import snapshot_download
import os

NYU_RAW_DIR = '/content/nyu_raw'

if not os.path.exists(NYU_RAW_DIR):
    print('Downloading NYU-D validation split from HuggingFace...')
    snapshot_download(
        repo_id='sayakpaul/nyu_depth_v2',
        repo_type='dataset',
        local_dir=NYU_RAW_DIR,
        ignore_patterns=['*train*']
    )
    print('Download complete.')
else:
    print('NYU-D already downloaded.')

os.system(f'ls {NYU_RAW_DIR}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Download complete.


0

In [ ]:
# Extract RGB images and GT depth maps from .tar -> .h5 files
import tarfile, h5py, glob, os
import numpy as np
from PIL import Image

NYU_RGB_DIR = '/content/nyu_rgb'
NYU_GT_DIR  = '/content/nyu_depth_gt'
os.makedirs(NYU_RGB_DIR, exist_ok=True)
os.makedirs(NYU_GT_DIR,  exist_ok=True)

MAX_IMAGES = 100  # set to 0 to extract all validation images

tar_files = sorted(glob.glob(f'{NYU_RAW_DIR}/**/*.tar', recursive=True))
print(f'Found {len(tar_files)} .tar file(s)')

count = 0
for tar_path in tar_files:
    with tarfile.open(tar_path, 'r') as tar:
        for member in tar.getmembers():
            if MAX_IMAGES > 0 and count >= MAX_IMAGES:
                break
            if not member.name.endswith('.h5'):
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            with h5py.File(f, 'r') as h5:
                # NYU-D .h5 layout: 'rgb' shape (3, H, W), 'depth' shape (H, W) in metres
                rgb   = np.array(h5['rgb'],   dtype=np.uint8)    # (3, H, W)
                depth = np.array(h5['depth'], dtype=np.float32)  # (H, W), metres

            stem = f'{count:05d}'
            Image.fromarray(rgb.transpose(1, 2, 0)).save(
                os.path.join(NYU_RGB_DIR, stem + '.jpg'))
            np.save(os.path.join(NYU_GT_DIR, stem + '.npy'), depth)
            count += 1

    if MAX_IMAGES > 0 and count >= MAX_IMAGES:
        break

print(f'Extracted {count} image pairs.')
print(f'  RGB   -> {NYU_RGB_DIR}')
print(f'  Depth -> {NYU_GT_DIR}')

Found 2 .tar file(s)
Extracted 100 image pairs.
  RGB   -> /content/nyu_rgb
  Depth -> /content/nyu_depth_gt


## 4. Load Model and Run DAV2 Inference (NYU-D)

Runs the metric-depth model on each RGB image and saves the predicted depth as `.npy`.

**Note on depth convention:**
- Model output: depth in metres, **larger = farther**
- NYU-D GT: also in metres, **larger = farther**
- `evaluate_metrics.py` convention: **larger = obstacle** (closer to camera)
- Therefore we **invert** both prediction and GT before evaluation (Step 5).

In [ ]:
import torch
from depth_anything_v2.dpt import DepthAnythingV2  # loaded from metric_depth/ via sys.path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
}

model = DepthAnythingV2(**{**model_configs['vits'], 'max_depth': 20})
model.load_state_dict(torch.load(CKPT, map_location='cpu'))
model = model.to(DEVICE).eval()
print(f'Model loaded on {DEVICE}')

Model loaded on cuda


In [ ]:
import glob, os
import numpy as np
from PIL import Image

PRED_DIR = './predictions/nyu'
GT_DIR   = './gt_depth/nyu'
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(GT_DIR,   exist_ok=True)

img_files = sorted(glob.glob(f'{NYU_RGB_DIR}/*.jpg'))
gt_files  = sorted(glob.glob(f'{NYU_GT_DIR}/*.npy'))
print(f'Running inference on {len(img_files)} images...')

for i, img_path in enumerate(img_files):
    fname = os.path.splitext(os.path.basename(img_path))[0] + '.npy'

    # Save prediction
    raw_img = np.array(Image.open(img_path).convert('RGB'))
    with torch.no_grad():
        depth_pred = model.infer_image(raw_img)  # metres, larger=farther
    np.save(os.path.join(PRED_DIR, fname), depth_pred.astype(np.float32))

    # Copy GT to expected location
    gt_src = os.path.join(NYU_GT_DIR, fname)
    gt_dst = os.path.join(GT_DIR, fname)
    if os.path.exists(gt_src) and not os.path.exists(gt_dst):
        import shutil
        shutil.copy(gt_src, gt_dst)

    if i % 10 == 0:
        print(f'  [{i+1:3d}/{len(img_files)}] {fname}  pred range=[{depth_pred.min():.2f}, {depth_pred.max():.2f}] m')

print(f'\nDone. Predictions saved to {PRED_DIR}')

Running inference on 100 images...
  [  1/100] 00000.npy  pred range=[1.80, 5.10] m
  [ 11/100] 00010.npy  pred range=[2.68, 5.64] m
  [ 21/100] 00020.npy  pred range=[0.97, 5.97] m
  [ 31/100] 00030.npy  pred range=[1.32, 2.13] m
  [ 41/100] 00040.npy  pred range=[1.43, 19.31] m
  [ 51/100] 00050.npy  pred range=[1.25, 4.09] m
  [ 61/100] 00060.npy  pred range=[1.85, 7.13] m
  [ 71/100] 00070.npy  pred range=[1.29, 3.38] m
  [ 81/100] 00080.npy  pred range=[0.87, 3.57] m
  [ 91/100] 00090.npy  pred range=[1.21, 4.52] m

Done. Predictions saved to ./predictions/nyu


## 5. Evaluate Metrics (NYU-D)

Computes **Free-space IoU** and **Obstacle Precision/Recall** using the same threshold convention as `nav.py` (`CLOSE_THRESHOLD = 0.4`).

Both prediction and GT are **inverted** before evaluation so that small depth values (close = obstacle) map to large normalised values, matching `evaluate_metrics.py`'s convention.
Invalid GT pixels (depth == 0) are excluded via a valid mask.

In [ ]:
import numpy as np
import glob, os, json

THRESHOLD = 0.4  # matches nav.py CLOSE_THRESHOLD

def normalize(x):
    mn, mx = x.min(), x.max()
    if mx - mn < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn)).astype(np.float32)

iou_list, prec_list, rec_list, per_image = [], [], [], []

for pred_path in sorted(glob.glob(f'{PRED_DIR}/*.npy')):
    fname   = os.path.basename(pred_path)
    gt_path = os.path.join(GT_DIR, fname)
    if not os.path.exists(gt_path):
        continue

    pred = np.load(pred_path).astype(np.float32)  # metres, larger=farther
    gt   = np.load(gt_path  ).astype(np.float32)  # metres, larger=farther

    # Exclude invalid GT pixels (depth == 0)
    valid = gt > 0
    if valid.sum() < 100:
        continue
    pred_v = pred[valid]
    gt_v   = gt[valid]

    # Invert: close (small metres) -> large value = obstacle
    # This matches evaluate_metrics.py convention: depth > threshold = obstacle
    pred_n = normalize(pred_v.max() - pred_v)
    gt_n   = normalize(gt_v.max()   - gt_v)

    # Free-space IoU  (free space = depth <= threshold)
    pred_free = pred_n <= THRESHOLD
    gt_free   = gt_n   <= THRESHOLD
    inter = np.logical_and(pred_free, gt_free).sum()
    union = np.logical_or (pred_free, gt_free).sum()
    iou   = float(inter) / float(union) if union > 0 else 1.0

    # Obstacle Precision / Recall  (obstacle = depth > threshold)
    pred_obs = pred_n > THRESHOLD
    gt_obs   = gt_n   > THRESHOLD
    TP = np.logical_and( pred_obs,  gt_obs).sum()
    FP = np.logical_and( pred_obs, ~gt_obs).sum()
    FN = np.logical_and(~pred_obs,  gt_obs).sum()
    precision = float(TP) / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = float(TP) / (TP + FN) if (TP + FN) > 0 else 0.0

    iou_list.append(iou)
    prec_list.append(precision)
    rec_list.append(recall)
    per_image.append({'file': fname, 'iou': round(iou, 4),
                      'precision': round(precision, 4), 'recall': round(recall, 4)})
    print(f'  {fname}  IoU={iou:.4f}  P={precision:.4f}  R={recall:.4f}')

summary_nyu = {
    'threshold':      THRESHOLD,
    'n_images':       len(iou_list),
    'mean_iou':       round(float(np.mean(iou_list)),  4),
    'mean_precision': round(float(np.mean(prec_list)), 4),
    'mean_recall':    round(float(np.mean(rec_list)),  4),
    'per_image':      per_image,
}

print(f'\n{"="*60}')
print(f'  Threshold      : {THRESHOLD}  (matches nav.py CLOSE_THRESHOLD)')
print(f'  Images         : {summary_nyu["n_images"]}')
print(f'  Mean IoU       : {summary_nyu["mean_iou"]:.4f}   (higher = better)')
print(f'  Mean Precision : {summary_nyu["mean_precision"]:.4f}   (higher = better)')
print(f'  Mean Recall    : {summary_nyu["mean_recall"]:.4f}   (higher = better, safety!)')
print(f'{"="*60}')

with open('results_nyu.json', 'w') as f:
    json.dump(summary_nyu, f, indent=2)
print('Saved results_nyu.json')

  00000.npy  IoU=0.7735  P=0.9473  R=0.7049
  00001.npy  IoU=0.5833  P=0.9541  R=0.8099
  00002.npy  IoU=0.7982  P=0.9816  R=0.9911
  00003.npy  IoU=0.7951  P=0.6064  R=0.6202
  00004.npy  IoU=0.5700  P=0.9659  R=1.0000
  00005.npy  IoU=0.7845  P=0.9983  R=0.9898
  00006.npy  IoU=0.8273  P=0.9976  R=0.9925
  00007.npy  IoU=0.8449  P=0.9967  R=0.9926
  00008.npy  IoU=0.9182  P=0.9973  R=0.9976
  00009.npy  IoU=0.5647  P=0.8519  R=0.4768
  00010.npy  IoU=0.1145  P=1.0000  R=0.5049
  00011.npy  IoU=0.0374  P=1.0000  R=0.5300
  00012.npy  IoU=0.9107  P=0.8959  R=0.8854
  00013.npy  IoU=0.0831  P=1.0000  R=0.3722
  00014.npy  IoU=0.2517  P=0.1045  R=1.0000
  00015.npy  IoU=0.6762  P=0.9503  R=0.6959
  00016.npy  IoU=0.5471  P=0.7292  R=0.9875
  00017.npy  IoU=0.7453  P=0.7579  R=0.9516
  00018.npy  IoU=0.8050  P=0.9173  R=0.9974
  00019.npy  IoU=0.1238  P=0.9575  R=0.7157
  00020.npy  IoU=0.8248  P=0.6317  R=0.9979
  00021.npy  IoU=0.8136  P=0.4840  R=0.9306
  00022.npy  IoU=0.8595  P=0.908

## 6. KITTI Evaluation (Optional)

KITTI requires a manual download due to its license agreement.
Mount your Google Drive containing the KITTI data, then uncomment and run the cells below.

KITTI depth convention: 16-bit PNG, divide by 256 to get metres. Same inversion logic applies.

In [ ]:
# Uncomment after mounting Drive with KITTI data

# from google.colab import drive
# drive.mount('/content/drive')

# KITTI_RGB_DIR = '/content/drive/MyDrive/kitti/rgb'
# KITTI_GT_DIR  = '/content/drive/MyDrive/kitti/depth_gt'

# os.system('python prepare_eval_data.py '
#           '--dataset kitti '
#           f'--img-dir {KITTI_RGB_DIR} '
#           f'--gt-dir  {KITTI_GT_DIR} '
#           '--encoder vits --max-images 100')

# os.system('python evaluate_metrics.py '
#           '--pred_dir ./predictions/kitti '
#           '--gt_dir   ./gt_depth/kitti '
#           '--threshold 0.4 '
#           '--output results_kitti.json')

print('KITTI cells are commented out. Uncomment after mounting Drive with KITTI data.')

KITTI cells are commented out. Uncomment after mounting Drive with KITTI data.


## 7. Final Summary Table and Download Results

In [ ]:
import json, os

results = {}
for dataset, fname in [('NYU-D', 'results_nyu.json'), ('KITTI', 'results_kitti.json')]:
    if os.path.exists(fname):
        with open(fname) as f:
            results[dataset] = json.load(f)

if not results:
    print('No results files found yet. Run the evaluation cells above first.')
else:
    print('=' * 62)
    print(f'{"Dataset":<10} {"N":>6} {"IoU":>8} {"Precision":>10} {"Recall":>8}')
    print('-' * 62)
    for name, r in results.items():
        print(f'{name:<10} {r["n_images"]:>6} {r["mean_iou"]:>8.4f} '
              f'{r["mean_precision"]:>10.4f} {r["mean_recall"]:>8.4f}')
    print('=' * 62)
    print('Recall is most safety-critical (missed obstacles = dangerous)')

Dataset         N      IoU  Precision   Recall
--------------------------------------------------------------
NYU-D         100   0.5661     0.8887   0.8310
Recall is most safety-critical (missed obstacles = dangerous)


In [ ]:
# Download results JSON files to your local machine
from google.colab import files

for fname in ['results_nyu.json', 'results_kitti.json']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloaded {fname}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded results_nyu.json
